# 🧪 Teste A/B — Qual Landing Page Performa Melhor?
**Análise estatística comparando tempo de permanência entre duas versões de página web**

---

## 🎯 Contexto de Negócio

Uma empresa digital testou duas versões de sua landing page (Página A e Página B) para descobrir qual retém os visitantes por mais tempo. O tempo de permanência é um indicador importante de engajamento — quanto mais tempo o usuário fica, maior a chance de conversão.

**Pergunta de negócio:** *A nova versão da página (B) é estatisticamente melhor que a versão atual (A)?*

---

## 0. Imports e Configurações

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import warnings
import os

warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (12, 5)

COR_A = '#1a6b3c'
COR_B = '#f5c518'
COR_DESTAQUE = '#d62728'

os.makedirs('../images', exist_ok=True)
print('✅ Ambiente configurado!')

## 1. Carregamento e Exploração dos Dados

In [ ]:
df = pd.read_csv(r'C:\Users\roney\OneDrive\Área de Trabalho\Projetos\Projeto_Teste_AB\archive\web_page_data.csv')

print(f'✅ Dataset carregado: {df.shape[0]} registros | {df["Page"].nunique()} grupos')
print(f'\nDistribuição por grupo:')
print(df['Page'].value_counts())
print(f'\nEstatísticas gerais:')
print(df.groupby('Page')['Time'].describe().round(2))
df.head(10)

## 2. Análise Exploratória (EDA)

### 2.1 Distribuição do Tempo por Página

In [ ]:
grupo_a = df[df['Page'] == 'Page A']['Time']
grupo_b = df[df['Page'] == 'Page B']['Time']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Histograma
axes[0].hist(grupo_a, bins=8, color=COR_A, alpha=0.7, label='Página A', edgecolor='white')
axes[0].hist(grupo_b, bins=8, color=COR_B, alpha=0.7, label='Página B', edgecolor='white')
axes[0].axvline(grupo_a.mean(), color=COR_A, linestyle='--', linewidth=2)
axes[0].axvline(grupo_b.mean(), color=COR_B, linestyle='--', linewidth=2)
axes[0].set_title('Distribuição do Tempo\npor Página', fontweight='bold')
axes[0].set_xlabel('Tempo (minutos)')
axes[0].set_ylabel('Frequência')
axes[0].legend()

# Boxplot
data_box = [grupo_a.values, grupo_b.values]
bp = axes[1].boxplot(data_box, patch_artist=True, labels=['Página A', 'Página B'])
bp['boxes'][0].set_facecolor(COR_A)
bp['boxes'][1].set_facecolor(COR_B)
for median in bp['medians']:
    median.set_color('white')
    median.set_linewidth(2)
axes[1].set_title('Boxplot — Tempo\npor Página', fontweight='bold')
axes[1].set_ylabel('Tempo (minutos)')

# Barras com médias
medias = [grupo_a.mean(), grupo_b.mean()]
erros = [grupo_a.sem(), grupo_b.sem()]
bars = axes[2].bar(['Página A', 'Página B'], medias, color=[COR_A, COR_B],
                   yerr=erros, capsize=8, edgecolor='white', error_kw={'linewidth': 2})
for bar, val in zip(bars, medias):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                 f'{val:.2f} min', ha='center', fontweight='bold', fontsize=11)
axes[2].set_title('Média de Tempo\n(com intervalo de confiança)', fontweight='bold')
axes[2].set_ylabel('Tempo médio (minutos)')
axes[2].set_ylim(0, max(medias) * 1.4)

plt.suptitle('Análise Exploratória — Tempo de Permanência por Página', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../images/01_eda_distribuicao.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n📊 Resumo:')
print(f'Página A — Média: {grupo_a.mean():.2f} min | Mediana: {grupo_a.median():.2f} min | Desvio padrão: {grupo_a.std():.2f}')
print(f'Página B — Média: {grupo_b.mean():.2f} min | Mediana: {grupo_b.median():.2f} min | Desvio padrão: {grupo_b.std():.2f}')
print(f'\nDiferença absoluta: {abs(grupo_b.mean() - grupo_a.mean()):.2f} min')
print(f'Diferença relativa: {abs(grupo_b.mean() - grupo_a.mean()) / grupo_a.mean() * 100:.1f}%')

## 3. Teste de Hipóteses

Antes de qualquer teste estatístico, precisamos definir claramente nossas hipóteses:

| | Hipótese |
|--|--|
| **H₀ (nula)** | Não há diferença entre o tempo médio das páginas A e B |
| **H₁ (alternativa)** | Existe diferença significativa entre os tempos médios |
| **Nível de significância (α)** | 0.05 (95% de confiança) |

> **Regra de decisão:** Se p-value < 0.05 → rejeitamos H₀ → a diferença é estatisticamente significativa

### 3.1 Verificação de Normalidade (Shapiro-Wilk)

In [ ]:
stat_a, p_norm_a = stats.shapiro(grupo_a)
stat_b, p_norm_b = stats.shapiro(grupo_b)

print('🔍 Teste de Normalidade — Shapiro-Wilk')
print(f'Página A: estatística={stat_a:.4f} | p-value={p_norm_a:.4f} → {"Normal ✅" if p_norm_a > 0.05 else "Não Normal ⚠️"}')
print(f'Página B: estatística={stat_b:.4f} | p-value={p_norm_b:.4f} → {"Normal ✅" if p_norm_b > 0.05 else "Não Normal ⚠️"}')

normal = p_norm_a > 0.05 and p_norm_b > 0.05
print(f'\n→ Usar teste paramétrico (t-test): {"Sim" if normal else "Não — usaremos Mann-Whitney"}')

# QQ Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
stats.probplot(grupo_a, dist='norm', plot=axes[0])
axes[0].set_title('QQ Plot — Página A', fontweight='bold')
axes[0].get_lines()[0].set(color=COR_A, markersize=8)
stats.probplot(grupo_b, dist='norm', plot=axes[1])
axes[1].set_title('QQ Plot — Página B', fontweight='bold')
axes[1].get_lines()[0].set(color=COR_B, markersize=8)
plt.tight_layout()
plt.savefig('../images/02_normalidade_qqplot.png', dpi=150)
plt.show()

### 3.2 Teste Estatístico

In [ ]:
alpha = 0.05

if normal:
    # Teste t de Student (paramétrico)
    stat, p_value = stats.ttest_ind(grupo_a, grupo_b)
    nome_teste = 'T-Test de Student (paramétrico)'
else:
    # Mann-Whitney (não paramétrico)
    stat, p_value = stats.mannwhitneyu(grupo_a, grupo_b, alternative='two-sided')
    nome_teste = 'Mann-Whitney U (não paramétrico)'

rejeita_h0 = p_value < alpha

print(f'🧪 Teste utilizado: {nome_teste}')
print(f'Estatística: {stat:.4f}')
print(f'P-value: {p_value:.4f}')
print(f'Nível de significância (α): {alpha}')
print()
if rejeita_h0:
    melhor = 'Página B' if grupo_b.mean() > grupo_a.mean() else 'Página A'
    print(f'✅ RESULTADO: Rejeitamos H₀ — A diferença É estatisticamente significativa!')
    print(f'   → {melhor} performa melhor com {alpha*100:.0f}% de nível de significância.')
else:
    print(f'❌ RESULTADO: Não rejeitamos H₀ — A diferença NÃO é estatisticamente significativa.')
    print(f'   → Não há evidência suficiente para afirmar que uma página é melhor que a outra.')

### 3.3 Visualização do P-Value

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

x = np.linspace(-4, 4, 300)
y = stats.norm.pdf(x)

ax.plot(x, y, color='gray', linewidth=2)
ax.fill_between(x, y, where=(x <= -1.96), color=COR_DESTAQUE, alpha=0.4, label='Região crítica (α/2 = 0.025)')
ax.fill_between(x, y, where=(x >= 1.96), color=COR_DESTAQUE, alpha=0.4)
ax.fill_between(x, y, where=((x > -1.96) & (x < 1.96)), color='#2196F3', alpha=0.15, label='Região de não rejeição')

# Marca o p-value
z_score = stats.norm.ppf(1 - p_value/2) if rejeita_h0 else stats.norm.ppf(p_value/2)
ax.axvline(abs(z_score), color=COR_A, linewidth=2.5, linestyle='--', label=f'Estatística do teste')
ax.axvline(-abs(z_score), color=COR_A, linewidth=2.5, linestyle='--')
ax.axvline(1.96, color=COR_DESTAQUE, linewidth=1.5, linestyle=':')
ax.axvline(-1.96, color=COR_DESTAQUE, linewidth=1.5, linestyle=':')

ax.set_title(f'Distribuição Normal — Teste de Hipóteses\np-value = {p_value:.4f} | α = {alpha} | {"H₀ Rejeitada ✅" if rejeita_h0 else "H₀ Não Rejeitada ❌"}',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Z-score')
ax.set_ylabel('Densidade')
ax.legend()
plt.tight_layout()
plt.savefig('../images/03_teste_hipotese.png', dpi=150)
plt.show()

## 4. Simulação com Dataset Sintético

Para reforçar o aprendizado, vamos simular um experimento maior com **1.000 usuários por grupo**, controlando os parâmetros do experimento.

In [ ]:
np.random.seed(42)
n = 1000

# Simulando: Página B tem média 15% maior que Página A
sintetico_a = np.random.normal(loc=3.0, scale=1.2, size=n)
sintetico_b = np.random.normal(loc=3.45, scale=1.3, size=n)

sintetico_a = np.clip(sintetico_a, 0.1, None)
sintetico_b = np.clip(sintetico_b, 0.1, None)

df_sint = pd.DataFrame({
    'Page': ['Page A'] * n + ['Page B'] * n,
    'Time': np.concatenate([sintetico_a, sintetico_b])
})

stat_sint, p_sint = stats.ttest_ind(sintetico_a, sintetico_b)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribuição sintética
axes[0].hist(sintetico_a, bins=40, color=COR_A, alpha=0.6, label=f'Página A (μ={sintetico_a.mean():.2f})', edgecolor='white')
axes[0].hist(sintetico_b, bins=40, color=COR_B, alpha=0.6, label=f'Página B (μ={sintetico_b.mean():.2f})', edgecolor='white')
axes[0].axvline(sintetico_a.mean(), color=COR_A, linestyle='--', linewidth=2)
axes[0].axvline(sintetico_b.mean(), color=COR_B, linestyle='--', linewidth=2)
axes[0].set_title(f'Dataset Sintético — {n} usuários/grupo\np-value={p_sint:.6f} | {"Significativo ✅" if p_sint < 0.05 else "Não Significativo ❌"}', fontweight='bold')
axes[0].set_xlabel('Tempo (minutos)')
axes[0].set_ylabel('Frequência')
axes[0].legend()

# Comparação real vs sintético
categorias = ['Dataset Real\n(36 obs)', 'Dataset Sintético\n(2.000 obs)']
p_values = [p_value, p_sint]
cores_p = [COR_A if p < 0.05 else COR_DESTAQUE for p in p_values]
bars = axes[1].bar(categorias, p_values, color=cores_p, edgecolor='white', width=0.4)
axes[1].axhline(0.05, color=COR_DESTAQUE, linestyle='--', linewidth=2, label='α = 0.05')
for bar, val in zip(bars, p_values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'p={val:.4f}', ha='center', fontweight='bold')
axes[1].set_title('Comparação de P-Values\nReal vs Sintético', fontweight='bold')
axes[1].set_ylabel('P-Value')
axes[1].legend()
axes[1].set_ylim(0, max(p_values) * 1.5)

plt.suptitle('Impacto do Tamanho da Amostra no Teste A/B', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../images/04_sintetico_vs_real.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n💡 Lição importante:')
print(f'Com {36} observações → p-value = {p_value:.4f} ({"Significativo" if p_value < 0.05 else "Não significativo"})')
print(f'Com {n*2} observações → p-value = {p_sint:.6f} ({"Significativo" if p_sint < 0.05 else "Não significativo"})')
print(f'\n→ Tamanho de amostra importa! Com poucos dados, efeitos reais podem passar despercebidos.')

## 5. Poder Estatístico e Tamanho de Amostra

Uma análise profissional de A/B Test sempre inclui o cálculo de **quantos usuários** precisamos para detectar um efeito real.

In [ ]:
from scipy.stats import norm

def calcular_tamanho_amostra(media_a, desvio_a, efeito_minimo_pct, alpha=0.05, poder=0.80):
    """Calcula o tamanho mínimo de amostra para detectar um efeito."""
    z_alpha = norm.ppf(1 - alpha/2)
    z_beta = norm.ppf(poder)
    delta = media_a * efeito_minimo_pct
    n = ((z_alpha + z_beta) * desvio_a / delta) ** 2
    return int(np.ceil(n))

media_ref = grupo_a.mean()
desvio_ref = grupo_a.std()

efeitos = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
tamanhos = [calcular_tamanho_amostra(media_ref, desvio_ref, e) for e in efeitos]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar([f'{int(e*100)}%' for e in efeitos], tamanhos, color=COR_A, edgecolor='white')
ax.axhline(18, color=COR_DESTAQUE, linestyle='--', linewidth=2, label=f'Amostra atual (n=18/grupo)')
for bar, val in zip(bars, tamanhos):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            f'n={val}', ha='center', fontsize=10, fontweight='bold')
ax.set_title('Tamanho de Amostra Necessário por Tamanho de Efeito\n(α=0.05, poder=80%)', fontweight='bold')
ax.set_xlabel('Efeito mínimo detectável (% de melhoria)')
ax.set_ylabel('Usuários necessários por grupo')
ax.legend()
plt.tight_layout()
plt.savefig('../images/05_tamanho_amostra.png', dpi=150)
plt.show()

print('📋 Tabela de referência:')
for e, t in zip(efeitos, tamanhos):
    print(f'  Detectar melhoria de {int(e*100)}% → precisamos de {t} usuários por grupo ({t*2} total)')

---
## 📋 Conclusões e Recomendações

### Resultados do Teste

| Métrica | Página A | Página B |
|---------|----------|----------|
| N amostras | 18 | 18 |
| Tempo médio | - min | - min |
| Desvio padrão | - | - |
| P-value | - | - |
| Conclusão | - | - |

### Lições Aprendidas

1. **Tamanho de amostra importa:** Com apenas 36 observações, o teste tem baixo poder estatístico — efeitos reais podem não ser detectados.
2. **Normalidade guia o teste:** Verificar a distribuição antes de escolher o teste estatístico é fundamental.
3. **Significância ≠ relevância prática:** Um resultado estatisticamente significativo pode ter impacto de negócio pequeno.
4. **Dataset sintético:** Permite simular cenários controlados e entender o comportamento dos testes antes de ir para produção.

---
**Próximo passo:** Aplicar Teste A/B com dados reais de conversão (cliques, cadastros) integrando com API de analytics.